In [0]:
%sql
drop table if exists demo.silver.h_customer;

create table if not exists demo.silver.h_customer
(
  customer_hk string not null comment "MD5(customer_id)",
  customer_id string NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_h_customer primary key (customer_hk) 
)

In [0]:
%sql
drop table if exists demo.silver.s_customer;

create table demo.silver.s_customer(
  customer_hk string not null comment "MD5(customer_id)",
  hash_diff string not null comment "Hash of all columns",
  customer_id string not null,
  first_name string not null,
  last_name string not null,
  company string not null,
  city string not null,
  country string not null,
  phone_1 string not null,
  phone_2 string not null,
  email string not null,
  subscription_date string not null,
  website string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_customer primary key(customer_hk)
)

In [0]:
%sql
drop table if exists demo.silver.h_product;

create table if not exists demo.silver.h_product
(
  product_hk string not null comment "MD5(product_id)",
  product_id INT NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_h_product primary key (product_hk) 
)

In [0]:
%sql
drop table if exists demo.silver.s_product;

create table  demo.silver.s_product(
  product_hk string not null comment "MD5(product_id)",
  hash_diff string not null comment "Hash of all non-key columns",
  product_id INT not null,
  name string not null,
  description string not null,
  brand string not null,
  category string not null,
  price string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_product primary key(product_hk)
)

In [0]:
%sql
drop table if exists demo.silver.l_order;

create table demo.silver.l_order
(
  order_hk string not null comment "MD5(customer_id || product_id)", 
  customer_id string NOT NULL,
  product_id INT NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_order primary key(order_hk)
)

In [0]:
%sql
drop table if exists demo.silver.s_l_order;

create table demo.silver.s_l_order
(
  order_hk string not null comment "MD5(customer_id || product_id)", 
  hash_diff string not null comment "Hash of all columns",
  customer_id string NOT NULL,
  product_id INT NOT NULL,
  vendor string not null,
  order_date date not null,
  order_amount numeric(10,2) not null,
  currency string not null,
  status string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_l_order primary key(order_hk)
)


In [0]:
%sql
insert into demo.silver.h_customer
select 
  md5(customer_id) as customer_hk,
  customer_id,
  current_timestamp() as load_time,
  'customer-10000.csv' as record_source
from (
  select distinct customer_id from demo.bronze.customer
)

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
insert into demo.silver.s_customer 
select 
  md5(customer_id) as customer_hk,
  sha2(concat_ws(',',customer_id,first_name,last_name,company,city,country,phone_1,phone_2,email,subscription_date,website),256) as hash_diff,
  customer_id,
  first_name,
  last_name,
  company,
  city, 
  country,
  phone_1,
  phone_2,
  email,
  subscription_date,
  website,
  current_timestamp() as load_time,
  'customer-10000.csv' as record_source
from (
  select distinct customer_id, first_name, last_name,company, city, country, phone_1, phone_2, email,subscription_date, website from demo.bronze.customer
)

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
insert into demo.silver.h_product
select 
  md5(product_id) as product_hk,
  product_id,
  current_timestamp() as load_time,
  'product-10000.csv' as record_source
from (
  select distinct product_id from demo.bronze.product
)


num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
insert into demo.silver.s_product 
select 
  md5(cast(product_id as string)) as product_hk,
  sha2(concat_ws(',',product_id, Name, Description, Brand, Category, Price, Currency, Stock, EAN, Color, Size,Availability, Internal_ID),256) as hash_diff,
  product_id,
  Name,
  Description,
  Brand,
  Category,
  Price,
  current_timestamp() as load_time,
  'product-10000.csv' as record_source
from (
  select distinct product_id, Name, Description,Brand,Category,Price,Currency,Stock,EAN,Color,Size,Availability,Internal_ID from demo.bronze.product
)

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

h_cust = spark.table("demo.bronze.customer")
h_prod = spark.table("demo.bronze.product")

l_order = h_cust.crossJoin(h_prod) \
  .withColumn('order_hk', md5(concat(col('customer_id'), col('product_id')))) \
  .withColumn('load_time', current_timestamp()) \
  .withColumn('record_source', lit('system')) \
  .select('order_hk', 'customer_id', col('product_id').cast('int').alias('product_id'), 'load_time', 'record_source')

l_order.write.mode('overwrite').saveAsTable('demo.silver.l_order')

In [0]:
from pyspark.sql.functions import *

l_order = spark.table('demo.silver.l_order').select('order_hk', 'customer_id', 'product_id', 'load_time', 'record_source').alias('l')
s_prod = spark.table('demo.silver.s_product').select('product_id', 'price', 'name').alias('p')
s_cust = spark.table('demo.silver.s_customer').select("customer_id").alias('c')

# Join l_order with s_prod and s_cust
s_l_order = l_order.join(s_prod, col('l.product_id') == col('p.product_id'), 'inner') \
    .join(s_cust, col('l.customer_id') == col('c.customer_id'), 'inner') \
    .withColumn('order_hk', md5(concat(col('l.customer_id'), col('l.product_id').cast('string')))) \
    .withColumn('vendor', lit('default_vendor')) \
    .withColumn('order_date', current_date()) \
    .withColumn('order_amount', col('p.price').cast('decimal(10,2)')) \
    .withColumn('currency', lit('USD')) \
    .withColumn('status', lit('shipped')) \
    .withColumn('load_time', current_timestamp()) \
    .withColumn('record_source', col('l.record_source')) \
    .withColumn('hash_diff', sha2( concat_ws(',', 'vendor', 'order_date', 'order_amount', 'currency', 'status', 'load_time', 'record_source'),  256)) \
    .select('order_hk', 'hash_diff', col('l.customer_id').alias('customer_id'), col('l.product_id').alias('product_id'), 'vendor', 'order_date', 'order_amount', 'currency', 'status', 'load_time', 'record_source')

display(s_l_order.limit(10))

s_l_order.write.mode('overwrite').saveAsTable('demo.silver.s_l_order')

order_hk,hash_diff,customer_id,product_id,vendor,order_date,order_amount,currency,status,load_time,record_source
6275cc9ad8cd187aa2526b8680aa9625,948dc331114175b0b5aa926895b475c68ae3cd9545389bb8412eff3c3566f3cc,EB54EF1154C3A78,1,default_vendor,2026-04-14,585.00,USD,shipped,2026-04-14T10:59:50.824Z,system
30bdde296862770b330dd54dc21f751c,aa6affb4dda361f6484524f88bf2a5eef7c5aa63e304e30146291c1091639789,10dAcafEBbA5FcA,2,default_vendor,2026-04-14,992.00,USD,shipped,2026-04-14T10:59:50.824Z,system
3bf9aa389775b5f107db6040dd9136a9,c9a71823a5e302f7fca6ceaf4ea6d476c12a16e2f07858e41c6205e480cfbf24,67DAB15Ebe4BE4a,3,default_vendor,2026-04-14,940.00,USD,shipped,2026-04-14T10:59:50.824Z,system
cf6ab26423633d9f7bf3a50acd7745e7,a6d9427fe70d08df343f259b1873d378be5d5827158d976c4951e424680d811b,6d350C5E5eDB4EE,4,default_vendor,2026-04-14,324.00,USD,shipped,2026-04-14T10:59:50.824Z,system
b3e50bf94801cbbe214683f7c3855f72,de137be8289effe1ddcd7b2b749f1add84cb8e3184bd827247dd04dba3cd29f9,5820deAdCF23EFe,5,default_vendor,2026-04-14,908.00,USD,shipped,2026-04-14T10:59:50.824Z,system
5eb628a4124c3d2db7dee78be1874148,eec29fc55aa2a41742db935007f986fb319f301ea3dcd97e0f1163ffcbdc1807,E1CDEaC63fDd5aA,6,default_vendor,2026-04-14,627.00,USD,shipped,2026-04-14T10:59:50.824Z,system
26dce5660158032c8bf94827d5d2fc91,1382dce630dd140f7adcd2ffba87598dedcde18e03d666ec651f994dbfdcbc29,3e1187fCcebC8d2,7,default_vendor,2026-04-14,385.00,USD,shipped,2026-04-14T10:59:50.824Z,system
094ef1fa293b6351dfbef11dccb02a6b,59b0caea85de6553b2c791270551ae6c8f2a3fc34693df2ba5a05a751317ebaf,47C5cEE243c9A7b,8,default_vendor,2026-04-14,296.00,USD,shipped,2026-04-14T10:59:50.824Z,system
9bdfa10dc74dd045f477086f2a0600e8,1673f3542e83cafcb576c1f9a6d33a1e732cea059afa42a9cf0acec7972968c1,cacaD68a5e4BF4b,9,default_vendor,2026-04-14,870.00,USD,shipped,2026-04-14T10:59:50.824Z,system
26ae499be11b007cb3295bedc0333ddd,0339b849167e303f0fd702ca7c76be7366fb165548bc5ab7efdedcf2a56baeec,436b9c41cfb1fa3,10,default_vendor,2026-04-14,602.00,USD,shipped,2026-04-14T10:59:50.824Z,system


Select the defalut cataelog and schem

In [0]:
%sql
describe table demo.silver.s_l_order

col_name,data_type,comment
order_hk,string,MD5(customer_id || product_id)
hash_diff,string,Hash of all columns
customer_id,string,null
product_id,int,null
vendor,string,null
order_date,date,null
order_amount,"decimal(10,0)",null
currency,string,null
status,string,null
load_time,timestamp,null
